In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 0.5 — PRE-FLIGHT VALIDITY CHECKS
# Inputs:  outputs/phase0_clean_data.parquet
#          outputs/01_universe_filtered.csv
# Outputs: outputs/phase05_stationarity_results.csv
#          outputs/phase05_fractional_diff_params.csv
#          outputs/phase05_validity_summary.csv
#          outputs/phase0_returns_matrix.parquet   ← ready for Phase 1
# ═══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

# Stats
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from scipy.stats import chi2

# Multiple testing
from statsmodels.stats.multitest import multipletests

Path("outputs").mkdir(exist_ok=True)

# ── Load Phase 0 outputs ───────────────────────────────────────────────────────
print("Loading Phase 0 outputs...")
df_raw        = pd.read_parquet("outputs/phase0_clean_data.parquet")
final_coins   = pd.read_csv("outputs/01_universe_filtered.csv")["coin_id"].tolist()
coin_meta     = pd.read_csv("outputs/03_coin_metadata_phase0.csv")

print(f"  Coins in universe : {len(final_coins)}")
print(f"  Raw data shape    : {df_raw.shape}")

# ── Build log-return matrix (T × N) ───────────────────────────────────────────
# Pivot to wide format, compute log returns, forward-fill ≤2-day gaps
prices = (
    df_raw[df_raw["coin_id"].isin(final_coins)]
    .pivot(index="date", columns="coin_id", values="close")
    .sort_index()
)
prices = prices.ffill(limit=2)                          # honour Phase 0 fill rules
log_returns = np.log(prices / prices.shift(1)).iloc[1:] # drop first NaN row

print(f"  Return matrix     : {log_returns.shape}  (days × coins)")


# ═══════════════════════════════════════════════════════════════════════════════
# 0.5.1  STATIONARITY TESTING — ADF + KPSS ON EVERY COIN
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 0.5.1 │ Stationarity Tests ──────────────────────────────────────")

def run_adf(series: pd.Series) -> dict:
    """
    ADF with lag order chosen by AIC (maxlag capped at 20).
    H0: unit root (non-stationary).
    Reject H0 → stationary.
    """
    clean = series.dropna()
    if len(clean) < 50:
        return {"adf_stat": np.nan, "adf_pval": np.nan, "adf_lags": np.nan,
                "adf_stationary": np.nan}
    result = adfuller(clean, maxlag=20, autolag="AIC", regression="c")
    return {
        "adf_stat"       : result[0],
        "adf_pval"       : result[1],
        "adf_lags"       : result[2],
        "adf_stationary" : result[1] < 0.05,   # True = stationary
    }


def run_kpss(series: pd.Series) -> dict:
    """
    KPSS with automatic bandwidth (Newey-West).
    H0: stationary.
    Reject H0 → non-stationary (unit root).
    """
    clean = series.dropna()
    if len(clean) < 50:
        return {"kpss_stat": np.nan, "kpss_pval": np.nan, "kpss_stationary": np.nan}
    stat, pval, _, _ = kpss(clean, regression="c", nlags="auto")
    return {
        "kpss_stat"       : stat,
        "kpss_pval"       : pval,
        "kpss_stationary" : pval >= 0.05,       # True = stationary
    }


# ─── Decision logic ───────────────────────────────────────────────────────────
def classify_stationarity(adf_stationary, kpss_stationary) -> str:
    """
    Both agree stationary    → STATIONARY
    Both agree non-stationary → UNIT_ROOT
    Conflicting              → CONFLICTING  (queue for ARFIMA)
    """
    if pd.isna(adf_stationary) or pd.isna(kpss_stationary):
        return "INSUFFICIENT_DATA"
    if adf_stationary and kpss_stationary:
        return "STATIONARY"
    if not adf_stationary and not kpss_stationary:
        return "UNIT_ROOT"
    return "CONFLICTING"


# ─── Run on all coins ─────────────────────────────────────────────────────────
stationarity_rows = []

for coin in final_coins:
    if coin not in log_returns.columns:
        continue
    series = log_returns[coin].dropna()
    adf  = run_adf(series)
    kpss_res = run_kpss(series)
    status = classify_stationarity(adf["adf_stationary"], kpss_res["kpss_stationary"])
    stationarity_rows.append({
        "coin_id": coin,
        **adf,
        **kpss_res,
        "stationarity_status": status,
    })

stat_df = pd.DataFrame(stationarity_rows)

# ─── Summary ──────────────────────────────────────────────────────────────────
status_counts = stat_df["stationarity_status"].value_counts()
print(f"\n  STATIONARY        : {status_counts.get('STATIONARY', 0)}")
print(f"  UNIT_ROOT         : {status_counts.get('UNIT_ROOT', 0)}")
print(f"  CONFLICTING       : {status_counts.get('CONFLICTING', 0)}  ← queue for ARFIMA")
print(f"  INSUFFICIENT_DATA : {status_counts.get('INSUFFICIENT_DATA', 0)}")

conflicting_coins = stat_df.loc[
    stat_df["stationarity_status"] == "CONFLICTING", "coin_id"
].tolist()

unit_root_coins = stat_df.loc[
    stat_df["stationarity_status"] == "UNIT_ROOT", "coin_id"
].tolist()

if unit_root_coins:
    print(f"\n  ⚠️  UNIT ROOT coins (will difference again in Phase 1): {unit_root_coins}")


# ═══════════════════════════════════════════════════════════════════════════════
# 0.5.2  FRACTIONAL INTEGRATION TEST — ARFIMA(d) FOR CONFLICTING COINS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 0.5.2 │ Fractional Integration (ARFIMA) ─────────────────────────")

def estimate_fractional_d(series: pd.Series, max_d_grid: int = 10) -> dict:
    """
    Estimates the fractional differencing parameter d via a grid search
    over ARFIMA(0, d, 0) models, selecting by AIC.

    Grid: d ∈ {0.0, 0.1, 0.2, ..., 1.0}

    Decision:
        d ≈ 0.0  → treat as I(0) — already stationary
        d ≈ 1.0  → treat as I(1) — use log returns
        0 < d < 0.5 → long memory — apply fractional differencing
    """
    clean = series.dropna()
    if len(clean) < 100:
        return {"d_estimate": np.nan, "d_aic": np.nan, "d_action": "INSUFFICIENT_DATA"}

    best_aic = np.inf
    best_d   = np.nan

    d_grid = np.arange(0.0, 1.1, 0.1)

    for d_val in d_grid:
        try:
            # ARIMA with fractional d: order=(0, d, 0) — statsmodels accepts float d
            model  = ARIMA(clean, order=(0, d_val, 0), trend="n")
            fitted = model.fit(method_kwargs={"warn_convergence": False})
            if fitted.aic < best_aic:
                best_aic = fitted.aic
                best_d   = d_val
        except Exception:
            continue

    # Classify action
    if np.isnan(best_d):
        action = "INSUFFICIENT_DATA"
    elif best_d <= 0.1:
        action = "TREAT_AS_I0"           # stationary — use as-is
    elif best_d >= 0.9:
        action = "TREAT_AS_I1"           # use log returns (already done)
    else:
        action = "FRACTIONAL_DIFF"       # 0.1 < d < 0.9 → apply fracdiff

    return {"d_estimate": round(best_d, 2), "d_aic": round(best_aic, 2), "d_action": action}


# ─── Fractional differencing transform ───────────────────────────────────────
def fracdiff(series: pd.Series, d: float, threshold: float = 1e-5) -> pd.Series:
    """
    Applies fractional differencing to a series.
    Uses a fixed-width window where weights drop below `threshold`.

    Jensen & Rahbek (2007) — finite memory approximation.
    """
    T      = len(series)
    # Compute weights: w_k = Π_{j=0}^{k-1} (j - d) / (j + 1)
    weights = [1.0]
    for k in range(1, T):
        w = -weights[-1] * (d - k + 1) / k
        if abs(w) < threshold:
            break
        weights.append(w)

    weights = np.array(weights)
    n_w     = len(weights)

    result = np.full(T, np.nan)
    for t in range(n_w - 1, T):
        result[t] = np.dot(weights, series.iloc[t : t - n_w if t - n_w >= 0 else None : -1])

    return pd.Series(result, index=series.index)


# ─── Run ARFIMA on conflicting coins ─────────────────────────────────────────
fracdiff_rows = []
fracdiff_series = {}  # coin_id → transformed series

if conflicting_coins:
    print(f"  Running ARFIMA on {len(conflicting_coins)} conflicting coins...")
    for coin in conflicting_coins:
        series = log_returns[coin].dropna()
        result = estimate_fractional_d(series)
        fracdiff_rows.append({"coin_id": coin, **result})

        if result["d_action"] == "FRACTIONAL_DIFF":
            d_val = result["d_estimate"]
            print(f"    {coin:>12} │ d={d_val:.2f} → applying fractional differencing")
            fracdiff_series[coin] = fracdiff(series, d=d_val)
        else:
            print(f"    {coin:>12} │ d={result['d_estimate']:.2f} → {result['d_action']}")

    fracdiff_df = pd.DataFrame(fracdiff_rows)

    action_counts = fracdiff_df["d_action"].value_counts()
    print(f"\n  TREAT_AS_I0      : {action_counts.get('TREAT_AS_I0', 0)}")
    print(f"  TREAT_AS_I1      : {action_counts.get('TREAT_AS_I1', 0)}")
    print(f"  FRACTIONAL_DIFF  : {action_counts.get('FRACTIONAL_DIFF', 0)}  ← replaced in return matrix")
else:
    print("  No conflicting coins — ARFIMA step skipped.")
    fracdiff_df = pd.DataFrame(columns=["coin_id", "d_estimate", "d_aic", "d_action"])


# ═══════════════════════════════════════════════════════════════════════════════
# 0.5.3  MULTIPLE TESTING CORRECTION — BENJAMINI-HOCHBERG FDR SETUP
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 0.5.3 │ Multiple Testing Framework ───────────────────────────────")

N          = len(final_coins)
n_pairs    = N * (N - 1) // 2
alpha      = 0.05
q_fdr      = 0.05
expected_fp = n_pairs * alpha

print(f"\n  Coins in universe    : {N}")
print(f"  Unique pairs         : {n_pairs:,}")
print(f"  Naïve α = 0.05 FP    : ~{expected_fp:,.0f}  ← why BH correction is mandatory")
print(f"  BH-FDR level (q)     : {q_fdr}")
print(f"  Expected false discoveries at q=0.05: ~{n_pairs * q_fdr * 0.05:.0f}")


def apply_bh_correction(pvalues: np.ndarray, q: float = 0.05) -> dict:
    """
    Benjamini-Hochberg FDR correction.

    Parameters
    ----------
    pvalues : array of raw p-values (one per test / pair)
    q       : target FDR level (default 0.05)

    Returns
    -------
    dict with:
        adjusted_pvalues  → BH-adjusted p-values
        reject            → bool array: True = significant after correction
        n_significant     → count of significant tests
        fdr_threshold     → the BH p-value threshold used
    """
    reject, pvals_corrected, _, _ = multipletests(
        pvalues, alpha=q, method="fdr_bh", is_sorted=False
    )
    bh_threshold = pvals_corrected[np.argsort(pvalues)].max() if reject.any() else 0.0
    return {
        "adjusted_pvalues" : pvals_corrected,
        "reject"           : reject,
        "n_significant"    : reject.sum(),
        "fdr_threshold"    : bh_threshold,
    }

# Demonstrate on the ADF p-values from this phase as a sanity check
adf_pvals = stat_df["adf_pval"].dropna().values
bh_demo   = apply_bh_correction(adf_pvals, q=q_fdr)
print(f"\n  BH demo on ADF p-values ({len(adf_pvals)} coins):")
print(f"    Significant pre-correction  (α=0.05) : {(adf_pvals < 0.05).sum()}")
print(f"    Significant post-BH (q=0.05)         : {bh_demo['n_significant']}")
print("  → apply_bh_correction() ready — import and use in Phases 2–6 on pair p-values")


# ═══════════════════════════════════════════════════════════════════════════════
# BUILD FINAL RETURN MATRIX — APPLY FRACDIFF OVERRIDES
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Building validated return matrix ───────────────────────────────────────")

returns_final = log_returns.copy()

# Replace fractionally-differenced series
for coin, fd_series in fracdiff_series.items():
    returns_final[coin] = fd_series
    print(f"  Replaced {coin} with fractionally-differenced series (d={fracdiff_df.loc[fracdiff_df['coin_id']==coin,'d_estimate'].values[0]})")

# Drop coins flagged as UNIT_ROOT (will be re-differenced in Phase 1)
if unit_root_coins:
    print(f"\n  ⚠️  Flagging unit-root coins for Phase 1 double-differencing: {unit_root_coins}")
    # Keep in matrix but tag in metadata — Phase 1 handles re-differencing

print(f"\n  Final return matrix shape : {returns_final.shape}")
print(f"  NaN rate                  : {returns_final.isna().mean().mean():.2%}")


# ═══════════════════════════════════════════════════════════════════════════════
# VALIDITY GATE — FAIL FAST BEFORE PHASE 1
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Validity Gate ───────────────────────────────────────────────────────────")

issues = []

# Check: too many non-stationary coins
pct_unit_root = len(unit_root_coins) / N
if pct_unit_root > 0.10:
    issues.append(f"⚠️  {pct_unit_root:.0%} of coins have unit roots — check return computation")

# Check: too many conflicting coins unresolved
unresolved = fracdiff_df[fracdiff_df["d_action"] == "INSUFFICIENT_DATA"].shape[0] if len(fracdiff_df) else 0
if unresolved > 5:
    issues.append(f"⚠️  {unresolved} conflicting coins couldn't be resolved — inspect manually")

# Check: NaN rate in return matrix
nan_rate = returns_final.isna().mean().mean()
if nan_rate > 0.05:
    issues.append(f"⚠️  Return matrix NaN rate = {nan_rate:.1%} — exceeds 5% threshold")

if issues:
    print("\n  Issues found:")
    for issue in issues:
        print(f"    {issue}")
else:
    print("  ✅ All validity checks passed — safe to proceed to Phase 1")


# ═══════════════════════════════════════════════════════════════════════════════
# SAVE OUTPUTS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Saving outputs ──────────────────────────────────────────────────────────")

# 1. Stationarity results
stat_df.to_csv("outputs/phase05_stationarity_results.csv", index=False)
print("  ✅ phase05_stationarity_results.csv  → ADF + KPSS results per coin")

# 2. Fractional differencing params
fracdiff_df.to_csv("outputs/phase05_fractional_diff_params.csv", index=False)
print("  ✅ phase05_fractional_diff_params.csv → d estimates for conflicting coins")

# 3. Validity summary
validity_rows = []
for _, row in stat_df.iterrows():
    coin = row["coin_id"]
    fd_row = fracdiff_df[fracdiff_df["coin_id"] == coin].iloc[0] if coin in fracdiff_df["coin_id"].values else {}
    validity_rows.append({
        "coin_id"              : coin,
        "stationarity_status"  : row["stationarity_status"],
        "adf_pval"             : row["adf_pval"],
        "kpss_pval"            : row["kpss_pval"],
        "d_estimate"           : fd_row.get("d_estimate", np.nan),
        "d_action"             : fd_row.get("d_action", "N/A"),
        "ready_for_phase1"     : row["stationarity_status"] in ("STATIONARY", "CONFLICTING"),
        "needs_re_differencing": row["stationarity_status"] == "UNIT_ROOT",
    })

validity_df = pd.DataFrame(validity_rows)
validity_df.to_csv("outputs/phase05_validity_summary.csv", index=False)
print("  ✅ phase05_validity_summary.csv      → per-coin readiness flags")

# 4. Validated return matrix — the key Phase 1 input
returns_final.to_parquet("outputs/phase0_returns_matrix.parquet")
print("  ✅ phase0_returns_matrix.parquet     → (T × N) validated log-return matrix")

# ─── Final status ─────────────────────────────────────────────────────────────
print(f"""
╔══════════════════════════════════════════════════════════════╗
║  PHASE 0.5 COMPLETE                                          ║
╠══════════════════════════════════════════════════════════════╣
║  Coins tested             : {N:<4}                              ║
║  Stationary (ADF+KPSS)    : {status_counts.get('STATIONARY', 0):<4}                              ║
║  Unit root                : {status_counts.get('UNIT_ROOT', 0):<4}  → flag for Phase 1           ║
║  Conflicting → ARFIMA     : {len(conflicting_coins):<4}                              ║
║    ↳ Fractionally diff'd  : {len(fracdiff_series):<4}  → overrides in return matrix  ║
║  Pairs to test (total)    : {n_pairs:,}                        ║
║  BH-FDR q level           : 0.05  (apply_bh_correction ready) ║
║  Return matrix shape      : {returns_final.shape[0]} × {returns_final.shape[1]}                     ║
╚══════════════════════════════════════════════════════════════╝
""")

Loading Phase 0 outputs...
  Coins in universe : 125
  Raw data shape    : (87609, 38)
  Return matrix     : (729, 125)  (days × coins)

── Phase 0.5.1 │ Stationarity Tests ──────────────────────────────────────


C:\Users\shekh\AppData\Local\Temp\ipykernel_9928\226522734.py:83: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  stat, pval, _, _ = kpss(clean, regression="c", nlags="auto")
C:\Users\shekh\AppData\Local\Temp\ipykernel_9928\226522734.py:83: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  stat, pval, _, _ = kpss(clean, regression="c", nlags="auto")
C:\Users\shekh\AppData\Local\Temp\ipykernel_9928\226522734.py:83: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  stat, pval, _, _ = kpss(clean, regression="c", nlags="auto")
C:\Users\shekh\AppData\Local\Temp\ipykernel_9928\226522734.py:83: InterpolationWarning: The test statistic is ou


  STATIONARY        : 125
  UNIT_ROOT         : 0
  CONFLICTING       : 0  ← queue for ARFIMA
  INSUFFICIENT_DATA : 0

── Phase 0.5.2 │ Fractional Integration (ARFIMA) ─────────────────────────
  No conflicting coins — ARFIMA step skipped.

── Phase 0.5.3 │ Multiple Testing Framework ───────────────────────────────

  Coins in universe    : 125
  Unique pairs         : 7,750
  Naïve α = 0.05 FP    : ~388  ← why BH correction is mandatory
  BH-FDR level (q)     : 0.05
  Expected false discoveries at q=0.05: ~19

  BH demo on ADF p-values (125 coins):
    Significant pre-correction  (α=0.05) : 125
    Significant post-BH (q=0.05)         : 125
  → apply_bh_correction() ready — import and use in Phases 2–6 on pair p-values

── Building validated return matrix ───────────────────────────────────────

  Final return matrix shape : (729, 125)
  NaN rate                  : 3.99%

── Validity Gate ───────────────────────────────────────────────────────────
  ✅ All validity checks passed — saf